# Chapter 7: Reconnaissance and OSINT

> "Know your enemy and know yourself and you can fight a hundred battles without disaster." Sun Tzu

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Distinguish passive from active reconnaissance and the legal implications of each.
2. Use open-source intelligence (OSINT) techniques responsibly.
3. Enumerate DNS records and interpret what they reveal.
4. Construct targeted search queries to surface exposed information.
5. Quantify an organization's external attack surface.

## Key Terms

- **OSINT**: Open-Source Intelligence.
- **DNS**: Domain Name System.
- **WHOIS**: Protocol for querying domain registration data.
- **Footprinting**: Building a profile of a target.
- **Attack surface**: The set of points where an attacker can attempt entry.

---

## 7.1 Passive vs Active Reconnaissance

Reconnaissance gathers information about a target before any direct interaction. **Passive**
reconnaissance uses only publicly available data and never touches the target's systems, which makes it
low risk and hard to detect. **Active** reconnaissance interacts directly, such as querying the
target's DNS servers or probing services, and may be observable or even regulated. The boundary matters
legally: passive collection of public data is generally permissible, while active probing can require
authorization.

## 7.2 Open-Source Intelligence

OSINT draws on search engines, public records, social media, certificate transparency logs, code
repositories, and job postings, which often reveal the technologies an organization uses. The
discipline is to combine many small, individually harmless facts into a useful picture. The same
techniques defenders use to monitor their own exposure are the ones attackers use to find a way in.

## 7.3 DNS and Domain Footprinting

The Domain Name System maps human-readable names to addresses and exposes a surprising amount of
structure. A records map to IPv4 addresses, MX records reveal mail infrastructure, TXT records often
contain SPF and verification data, and NS records name the authoritative servers. WHOIS provides
registration details, although privacy services now mask much of it. Certificate transparency logs can
reveal subdomains an organization may not have intended to publicize.

## 7.4 Search Operators

Search engines support operators that narrow results, such as restricting to a site, a file type, or a
URL pattern. Used responsibly against your own assets, these queries surface forgotten documents,
exposed configuration files, and login portals. The technique is sometimes called search-engine
footprinting, and it is a reminder that anything indexed is effectively public.

## 7.5 Why This Matters

Most successful intrusions begin with information the target gave away without realizing it. Mapping
your own attack surface the way an attacker would is one of the highest-value defensive exercises an
organization can perform.

## 7.6 News in Focus

Repeated incidents involving publicly accessible cloud storage buckets, where misconfigured permissions
exposed sensitive files to anyone who found the URL, demonstrate how reconnaissance against an external
attack surface can yield serious data exposure without any exploitation of a software flaw.

## 7.7 Worked Example: Attack Surface Scoring

The code models a small external footprint and computes the share of undocumented shadow assets and an
overall exposure score, the kind of metric used to prioritize cleanup.


In [1]:
from dataclasses import dataclass
from typing import List

@dataclass
class Asset:
    ip: str
    hostname: str
    services: List[str]
    documented: bool
    outdated: bool

risky_services = {"ftp", "rdp", "telnet", "database", "admin_panel"}

def score(assets):
    total = len(assets)
    shadow = sum(1 for a in assets if not a.documented)
    points = 0
    maxpoints = 0
    rows = []
    for a in assets:
        s = 0
        m = len(a.services) * 2 + 4
        s += sum(2 for svc in a.services if svc in risky_services)
        if not a.documented: s += 2
        if a.outdated: s += 2
        points += s
        maxpoints += m
        rows.append((a.ip, a.hostname, s, m, a.documented, a.outdated))
    exposure = round(points / maxpoints * 100, 1)
    shadow_pct = round(shadow / total * 100, 1)
    level = "Critical" if exposure >= 60 else "High" if exposure >= 40 else "Moderate" if exposure >= 20 else "Low"
    return total, shadow, shadow_pct, exposure, level, rows

assets = [
    Asset("203.0.113.10", "www.example.com", ["https", "http"], True, False),
    Asset("203.0.113.11", "mail.example.com", ["https", "smtp"], True, True),
    Asset("203.0.113.20", "dev.example.com", ["http", "ftp"], False, True),
    Asset("203.0.113.21", "db-old.example.com", ["database", "rdp"], False, True),
]

total, shadow, shadow_pct, exposure, level, rows = score(assets)
print(f"Total assets:  {total}")
print(f"Shadow assets: {shadow} ({shadow_pct}%)")
print(f"Exposure:      {exposure}%  ->  Risk level: {level}\n")
print(f"{'IP':<16}{'Hostname':<24}{'Score':>6}")
print("-" * 46)
for ip, host, s, m, doc, old in rows:
    print(f"{ip:<16}{host:<24}{s:>3}/{m:<3}")


Total assets:  4
Shadow assets: 2 (50.0%)
Exposure:      50.0%  ->  Risk level: High

IP              Hostname                 Score
----------------------------------------------
203.0.113.10    www.example.com           0/8  
203.0.113.11    mail.example.com          2/8  
203.0.113.20    dev.example.com           6/8  
203.0.113.21    db-old.example.com        8/8  


## 7.8 Review Questions (MCQ)

**Q1.** Querying a target's authoritative DNS server directly is:
A. Always passive  B. Active reconnaissance  C. Illegal everywhere  D. Impossible

**Q2.** Which DNS record reveals mail infrastructure?
A. A  B. MX  C. NS  D. CNAME

**Q3.** An undocumented internet-facing host is best described as a:
A. Honeypot  B. Shadow asset  C. Bastion  D. Proxy

*Answers: Q1 B, Q2 B, Q3 B.*

## 7.9 Lab Assignment

For a domain you own or are explicitly authorized to assess, enumerate its public DNS records, identify
its mail provider from the MX records, and list any subdomains visible in certificate transparency logs.
Summarize your findings as an attack-surface inventory.

## References

```{bibliography}
:filter: docname in docnames
```
